# 在本機跑 GGUF 量化模型：用 llama-cpp-python 讓 Phi-3 產生 JSON

---

## 這堂課你會學到什麼

1. **llama.cpp、llama-cpp-python、GGUF、量化**分別是什麼，跟 transformers 有什麼不同
2. 為什麼載入前要**釋放 GPU 記憶體**，以及「顯存不夠」的錯誤長什麼樣子
3. 用 `Llama.from_pretrained` 載入 GGUF 模型，看懂每個參數
4. 用 `create_chat_completion` 產生回答，並讀懂**巢狀 dict** 的回傳值
5. 用 `response_format` 要求 **JSON** 輸出，再用 `json.loads` / `json.dumps` 整理
6. 為什麼最後要 `llm.close()`


## 0. 環境準備

```bash
pip install llama-cpp-python torch
```

- `llama-cpp-python` 要用 GPU 加速，安裝時需要用支援 CUDA 的方式編譯，做法依平台而異，請照官方說明安裝；只裝一般版本也能跑，但會是 CPU 模式，比較慢。
- `torch` 在這份程式裡只用來清理 GPU 快取（第 3 節）。
- 第一次執行 `from_pretrained` 會從 Hugging Face **下載模型檔並快取**，需要網路與硬碟空間。
- **記憶體需求（我查了 Hugging Face 倉庫頁面）：**

  | 檔案 | 大小 |
  |---|---|
  | `Phi-3-mini-4k-instruct-fp16.gguf` | 7.64 GB |
  | `Phi-3-mini-4k-instruct-q4.gguf` | 2.39 GB |

  這份教材用 q4，8 GB 等級的顯卡也放得下。

## 1. 先看整體：這條路和 transformers 有什麼不同？

「在本機跑 LLM」有兩條常見的路：

| | transformers（之前學的） | llama-cpp-python（這堂課） |
|---|---|---|
| 背後引擎 | PyTorch | llama.cpp（C++） |
| 模型格式 | 原始權重 | **GGUF**（量化壓縮檔） |
| 載入 | `AutoModelForCausalLM` | `Llama` |
| 生成 | `pipeline`、`model.generate()` | `create_chat_completion` 等 |

兩套工具**各自獨立，不能混用**。

```
 提示詞（字串）
    │  包成訊息 list：[{"role": "user", "content": ...}]
    ▼
 llm.create_chat_completion(...)   ← 內部自動套聊天模板、生成
    │
    ▼
 巢狀 dict（OpenAI 風格）：choices、usage、...
    │  ['choices'][0]['message']['content']
    ▼
 JSON 字串
    │  json.loads → json.dumps(indent=4)
    ▼
 排版整齊的 JSON
```

> 📌 **本節重點**
> - llama-cpp-python 是「在本機跑模型」的另一個函式庫，不是雲端 API。
> - 它的 `create_chat_completion` 刻意模仿 OpenAI 的格式，所以寫法看起來很像 OpenAI，但模型是在你自己的電腦上跑。

## 2. 匯入套件與名詞釐清

| 名稱 | 是什麼 |
|---|---|
| Llama（Meta） | 一個**模型家族**（例如 Llama 3） |
| llama.cpp | 用 C++ 寫的**推論引擎**：載入模型、一個一個 token 算出來 |
| llama-cpp-python | llama.cpp 的 **Python 包裝**，就是 `from llama_cpp.llama import Llama` |
| GGUF | llama.cpp 使用的**模型檔案格式**，常常是量化過的 |
| 量化（Quantization） | 把模型權重改用更少的位元儲存（如 4 位元），**檔案變小、更省記憶體**，代價是精度略降 |

> ⚠️ 程式裡的類別叫 `Llama`，但我們載入的是 **Phi-3**，不是 Meta 的 Llama。llama.cpp 後來支援很多模型，只要轉成 GGUF 就能跑。

| 套件 | 這裡的用途 |
|---|---|
| `torch` | 只用來呼叫 `torch.cuda.empty_cache()` 清 GPU 快取 |
| `gc` | Python 的垃圾回收 |
| `json` | 解析與重新排版 JSON |
| `Llama` | 載入 GGUF 模型並生成 |

In [ ]:
import torch
import gc
import json
#llama-cpp-python 是讓 Python 程式可以呼叫並使用高效能 C++ AI 推論引擎 llama.cpp 的專案套件
#支援 GGUF 格式：能夠載入經過量化（Quantization）壓縮的 GGUF 模型檔案
from llama_cpp.llama import Llama  

## 3. 釋放前面用過的 GPU 記憶體

如果同一個 notebook 之前載入過別的模型（例如 transformers 的 Phi-3），它還佔著 GPU 的記憶體，這裡就可能不夠用。

| 程式 | 作用 |
|---|---|
| `# del model, tokenizer, pipe` | 把 Python 變數名稱移除（目前是註解，要用的話自己打開，變數名稱要和你之前的一致） |
| `gc.collect()` | 要求 Python 回收**已經沒有人引用**的物件 |
| `torch.cuda.empty_cache()` | 把 PyTorch **沒在使用的 GPU 快取**還給系統 |

### ⚠️ 這兩行的限制（初學者常誤會）

- `gc.collect()` 只收「沒人引用」的物件。如果 `model`、`pipe` 這些變數**還存在**，物件就還被引用，**收不掉**。要先 `del` 它們（也就是把上面那行 `del` 打開），再 `gc.collect()`。
- `torch.cuda.empty_cache()` 只管 **PyTorch** 配置的快取。llama.cpp 用自己的方式配置 GPU 記憶體，所以它幫不上 llama.cpp 用掉的部分。
- 最乾淨的做法：**重啟 kernel**，只執行這份 notebook。可以用終端機的 `nvidia-smi` 查看顯存是否真的空出來。

In [ ]:
# del model, tokenizer, pipe  #把 Python 變數名稱移除
gc.collect()                  #要求 Python 做 garbage collection, 清理已經沒有被引用的 Python 物件
torch.cuda.empty_cache()      #釋放記憶體 把沒在使用的 GPU cache 釋放掉

> 📌 **本節重點**
> - 載入新模型前，先確定舊模型的記憶體已經放掉。
> - 光靠 `gc.collect()` 和 `empty_cache()` 不一定夠，**重啟 kernel** 最保險。

## 4. 設定要載入哪個模型：`REPO_ID` 與 `FILENAME`

Hugging Face 上的網址其實把兩個東西分開了：

```
https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/blob/main/Phi-3-mini-4k-instruct-q4.gguf
                       └────────────── repo_id ──────────────┘          └───────── filename ─────────┘
```

| 變數 | 意思 |
|---|---|
| `REPO_ID` | **倉庫**（像一個資料夾）的名稱，格式 `擁有者/倉庫名` |
| `FILENAME` | 倉庫裡**要抓哪個檔案** |

**為什麼 `FILENAME` 不能省？** ：

- `from_pretrained(repo_id, filename, ...)` 的 `filename` **沒有預設值**，一定要傳。
- 它是一個**萬用字元（glob）樣式**，用 `fnmatch` 比對倉庫裡的檔名：

  | 情況 | 結果 |
  |---|---|
  | 剛好比對到 1 個檔 | 下載並載入 |
  | 比對到多個檔（如 `"*.gguf"`） | `ValueError`，並列出有哪些檔 |
  | 一個都比對不到 | `ValueError`，並列出有哪些檔 |

- 這個倉庫有 **fp16 與 q4 兩個** `.gguf`；`"*q4.gguf"` 剛好只對到 q4 那一個。也可以直接寫完整檔名 `"Phi-3-mini-4k-instruct-q4.gguf"`，更明確。

In [ ]:
REPO_ID = "microsoft/Phi-3-mini-4k-instruct-gguf"
FILENAME = "*q4.gguf"     #filename：要抓哪個 .gguf 模型檔

## 5. 載入模型

```python
llm = Llama.from_pretrained(repo_id = REPO_ID, filename = FILENAME, n_gpu_layers = -1, n_ctx = 2048, verbose = False)
```

| 參數 | 意思 |
|---|---|
| `repo_id`、`filename` | 上一節設定的倉庫與檔案；`from_pretrained` 會**下載並快取**，之後就從快取載入 |
| `n_gpu_layers = -1` | 要放到 GPU 的層數；**`-1` = 全部都放 GPU**。顯存不夠時可以改成較小的數字，其餘的層留在 CPU（會變慢） |
| `n_ctx = 2048` | **上下文長度**：提示詞 + 生成的內容，加起來最多能容納的 token 數。調小可以省記憶體 |
| `verbose = False` | 不顯示 llama.cpp 的詳細日誌。**出錯時改成 `True`**，才看得到真正的原因 |

### 常見錯誤：`Failed to create llama_context`

這是 llama.cpp 配置記憶體失敗時的通用訊息，最常見的原因是**顯存不夠**（例如 8 GB 的卡去載 7.64 GB 的 fp16 檔）。處理順序：
1. 把 `verbose` 改成 `True`，看日誌裡是否有 out of memory；
2. 改用 q4 檔（2.39 GB）；
3. 確認前面的模型已釋放（第 3 節），必要時重啟 kernel；
4. 降低 `n_gpu_layers` 或 `n_ctx`。

In [ ]:
llm = Llama.from_pretrained(
    repo_id = REPO_ID,
    filename = FILENAME,
    n_gpu_layers = -1,
    n_ctx = 2048,
    verbose = False,
)

> 📌 **本節重點**
> - `repo_id` 是倉庫，`filename` 是倉庫裡的檔案；兩個都要給。
> - `n_gpu_layers`、`n_ctx` 是調整記憶體用量的兩個主要旋鈕。
> - 看不懂錯誤時，先把 `verbose=True`。

## 6. 準備提示詞

我們要請模型為 RPG 遊戲設計一個戰士，並且用 JSON 回答。提示詞裡**明確列出要有哪些欄位**：

```
name, class, level, attributes, equipment, inventory
```

這是很實用的技巧：只說「用 JSON 格式」，模型會自己決定結構，可能把欄位放在奇怪的層級（例如把 `inventory` 放到角色的外面）。明確列出 key，能讓結構比較穩定。

In [ ]:
prompt = "Create a warrior for an RPG in JSON format with the keys: name, class, level, attributes, equipment, inventory."

## 7. 產生回答：`create_chat_completion`

```python
llm.create_chat_completion(
    messages = [{"role": "user", "content": prompt}],
    response_format = {"type": "json_object"},
    temperature = 0,
)
```

| 參數 | 意思 |
|---|---|
| `messages` | 訊息 list，每則有 `role`（`user` / `assistant` / `system`）和 `content`。它會**自動套用聊天模板**（模板來源：你傳的 `chat_handler`、`chat_format`，或 GGUF 檔內建的模板） |
| `response_format = {"type": "json_object"}` | 要求輸出**限制為有效的 JSON**|
| `temperature = 0` | 不隨機：每一步都選機率最高的 token，結果穩定、可重現 |

### 回傳值是巢狀 dict（你的註解）

`create_chat_completion` 回傳的**不是單純的一段文字**，而是 OpenAI 風格的巢狀 dictionary：

```
{
  "choices": [                      ← list，通常只有 1 個元素
     {
       "message": {
          "role": "assistant",
          "content": "{...JSON 字串...}"      ← 我們要的答案
       },
       "finish_reason": "stop"      ← 結束原因
     }
  ],
  "usage": {...}                    ← token 使用量
}
```

所以取答案要一路往下：`['choices'][0]['message']['content']`。

> ⚠️ 注意 `content` 是一個**字串**（看起來像 JSON，但還只是文字），下一節才把它變成真正的 Python 物件。

In [ ]:
#llm.create_chat_completion(...) 回傳的不是單純一段文字，
#而是一個巢狀 dictionary，裡面包含模型回答、使用量、結束原因等資訊。
output = llm.create_chat_completion(
    messages = [
        {
            "role" : "user",
            "content" : prompt
        },
    ],
    response_format={"type" : "json_object"},
    temperature = 0,
)['choices'][0]['message']['content']

> 📌 **本節重點**
> - `messages` 傳訊息 list，模板它自己處理；不用像 transformers 手動 `apply_chat_template`。
> - 回傳是巢狀 dict，答案在 `['choices'][0]['message']['content']`，而且是字串。
> - `temperature=0` → 同樣的提示得到同樣的輸出。

## 8. 印出並整理成好讀的 JSON

```python
print(output)
json_output = json.dumps(json.loads(output), indent=4)
print(json_output)
```

| 步驟 | 作用 |
|---|---|
| `print(output)` | 印出模型**原始輸出**。縮排通常很亂，因為那是模型自己吐出的空白 |
| `json.loads(output)` | 把 JSON **字串**解析成 Python 的 dict / list。**如果不是有效 JSON，會出現 `JSONDecodeError`** |
| `json.dumps(..., indent=4)` | 把 Python 物件轉回 JSON 字串，並用 **4 個空格縮排**，整齊好讀 |

### 為什麼要繞一圈 `loads` 再 `dumps`？

兩個目的：
1. **驗證**：`json.loads` 成功，代表輸出真的是有效的 JSON。
2. **重新排版**：把亂七八糟的空白變成整齊的縮排。

### ⚠️ 兩個容易搞混的事

- `response_format` 保證的是**語法**有效，**不保證內容合理**。例如欄位放錯層級、數值怪異，語法上仍然是合法 JSON。
- 想在程式裡用這些資料（例如取出 `level`），要用 `json.loads` 的結果（dict），不是 `json.dumps` 的結果（字串）。

In [ ]:
print(output)
json_output = json.dumps(json.loads(output), indent=4)
print(json_output)

> 📌 **本節重點**
> - `loads`：字串 → Python 物件（也順便驗證格式）；`dumps(indent=4)`：Python 物件 → 好看的字串。
> - 結構化輸出要「**提示詞列出欄位** + `response_format` 限制語法」雙管齊下。

## 9. 關閉模型

```python
llm.close()
```

用完模型之後，**主動關閉**，讓它釋放記憶體。

### 為什麼不能放著不管？

如果不主動關閉，程式結束時 Python 會在清理階段呼叫 `Llama` 的 `__del__`，但這時 `llama_cpp` 模組裡的函式可能已經被先清空，就會出現這種訊息：

```
Exception ignored in: <function Llama.__del__ ...>
TypeError: 'NoneType' object is not callable
```

重點是開頭的 **`Exception ignored`**：Python 把它當成「已被忽略的例外」，**不影響結果**，只是難看。我查了這個套件的原始碼：`__del__` 會呼叫 `close()`，而 `close()` 內部只會真正釋放一次，所以**自己先呼叫 `llm.close()`** 通常能避免這個訊息。

### 原本的程式用了 `try / finally`

你的程式在 `__main__` 裡這樣寫：

```python
try:
    generator.generate_example(prompt)
finally:
    generator.close()
```

意思是：**不管中間成功或出錯，`finally` 裡的 `close()` 一定會執行**。在 notebook 裡我們一格一格跑，所以改成「最後一定要把這一格跑完」；如果前面的 cell 出錯了，也記得執行這一格，把模型關掉。

In [ ]:
llm.close()

> 📌 **本節重點**
> - 用完就 `llm.close()`，把記憶體還回去。
> - `try / finally`：確保「收尾動作」一定會執行。
> - `Exception ignored ...` 是結束時的雜訊，不是你程式的 bug。

## 10. 全課重點整理

### 流程回顧

| 步驟 | 程式 | 作用 |
|---|---|---|
| ① 釋放記憶體 | `gc.collect()`、`torch.cuda.empty_cache()` | 騰出 GPU 空間 |
| ② 設定模型 | `REPO_ID`、`FILENAME` | 倉庫 + 檔案（glob） |
| ③ 載入 | `Llama.from_pretrained(...)` | 下載、快取並載入 GGUF |
| ④ 生成 | `create_chat_completion(...)` | 傳訊息 list，取得巢狀 dict |
| ⑤ 取答案 | `['choices'][0]['message']['content']` | 得到 JSON 字串 |
| ⑥ 整理 | `json.loads` → `json.dumps(indent=4)` | 驗證並排版 |
| ⑦ 關閉 | `llm.close()` | 釋放記憶體 |

### 必記的 7 件事

1. llama-cpp-python 是**另一套**本機推論工具（llama.cpp + GGUF），跟 transformers 不能混用。
2. GGUF 常是**量化**過的：檔案小、省記憶體，精度略降。q4 約 2.39 GB，fp16 約 7.64 GB。
3. `repo_id` 是倉庫、`filename` 是倉庫裡的檔案（glob 樣式），**兩個都要給**，且要剛好比對到一個檔。
4. `n_gpu_layers=-1` 全放 GPU；`n_ctx` 決定上下文長度。記憶體不夠就調它們。
5. `create_chat_completion` 傳**訊息 list**，回傳**巢狀 dict**；答案在 `['choices'][0]['message']['content']`。
6. `response_format` 管**語法**，提示詞裡列出欄位管**結構**；`json.loads` 負責驗證。
7. 用完 `llm.close()`；`try / finally` 確保一定會執行。

### 常見錯誤

| 現象 | 可能原因與處理 |
|---|---|
| `ValueError: Failed to create llama_context` | 多半是顯存不夠。`verbose=True` 看原因；改用 q4；釋放前面的模型 / 重啟 kernel；降低 `n_gpu_layers`、`n_ctx` |
| `ValueError`（列出多個檔案或找不到檔案） | `filename` 的樣式比對到多個或零個檔 |
| `TypeError`（沒傳 `filename`） | `filename` 沒有預設值，一定要給 |
| `json.JSONDecodeError` | 輸出不是有效 JSON；可以檢查 `output`，或調整提示詞 |
| 結構奇怪（欄位放錯層級） | 語法有效不代表結構正確；在提示詞裡列出需要的 key |
| `Exception ignored in: ... __del__` | 程式結束時的雜訊；先呼叫 `llm.close()` |

### 動手練習

1. 把 `verbose` 改成 `True`，觀察載入時 llama.cpp 輸出了哪些資訊。
2. 把 `n_gpu_layers` 改成 `10`，比較速度與記憶體使用。
3. 把提示詞改成要求一個法師（wizard），並改變要求的 key。
4. 把提示詞裡的 `with the keys: ...` 拿掉，觀察輸出結構有沒有變得不穩定。
5. 把 `temperature` 改成 `1.0`，連續跑 3 次，看輸出是否不同。
6. 進階：用 `output` 解析出的 dict，印出角色的 `name` 與 `level`。

### 延伸方向

- `create_completion` / `llm(prompt)`：自己組好完整字串的純文字接續
- `generate(tokens)`：低階、逐 token 的控制
- `chat_format` 與 `chat_handler`：自訂聊天模板
- 比較 q4 與 fp16 的輸出品質與速度差異